# FER2013 — ResNet18 + CBAM + Compact Convolutional Transformer

Notebook chạy độc lập trên Kaggle, không cần thư mục `src/fer`.
Cấu hình mặc định **ResNet18 → CBAM → 49 token → 2 Transformer Encoder
(512 chiều, 8 heads) → Sequence Pooling → 7 lớp**, tổng **17.543.338 tham số**.

## Thí nghiệm CBAM

Run `combined` trước đạt validation accuracy 69.63%, test accuracy 68.84%
và test Macro F1 67.59%. Lần thử mới chỉ thêm CBAM vào cấu hình đó:

- CBAM đặt sau feature map `512 × 7 × 7` của ResNet18, trước flatten/tokenization.
- Channel Attention rồi Spatial Attention; reduction ratio 16, spatial kernel 7.
- Thêm 32.866 tham số, khoảng 0.19% so với model combined 17.510.472 tham số.
- Giữ 2 encoder, embedding 512, MixUp alpha 0.1/xác suất 0.3, weight decay 3e-4.
- Giữ split seed 42, learning rate, warmup, loss, augmentation và 50 epoch.
- Ghi cấu hình CBAM vào checkpoint để tạo lại đúng kiến trúc khi nạp.
- Dùng hai T4 bằng DataParallel khi khả dụng; có thể tắt để so tốc độ một GPU.

Chọn `EXPERIMENT='cbam'` để train model mới hoặc `'combined'` để đối chứng
với model không CBAM. Mỗi lần chạy train một preset. Đánh giá thay đổi bằng
validation trước, giữ test cho bước cuối. Chưa có accuracy thực nghiệm cho CBAM.

**Cách chạy:** gắn dataset FER-2013 có `train/` và `test/`, bật GPU và chạy
lần lượt các cell. Bật Internet nếu cần tải weights ResNet18 lần đầu.
Không cần tạo thêm thư mục `val/` hay di chuyển ảnh trên Kaggle.

In [ ]:
import json
import math
import os
import random
import sys
from time import perf_counter
from concurrent.futures import ThreadPoolExecutor
from contextlib import closing, nullcontext
from datetime import datetime, timezone
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
from sklearn.metrics import classification_report, confusion_matrix, f1_score
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, models, transforms
from tqdm.auto import tqdm

# Resolve local defaults from the checkout when launched from a nested notebook folder.
WORKING_DIR = Path.cwd()
PROJECT_ROOT = next(
    (directory for directory in (WORKING_DIR, *WORKING_DIR.parents)
     if (directory / 'pyproject.toml').is_file() and (directory / 'src' / 'fer').is_dir()),
    WORKING_DIR,
)

CLASS_NAMES = ('angry', 'disgust', 'fear', 'happy', 'neutral', 'sad', 'surprise')
NORMALIZE_MEAN = (0.485, 0.456, 0.406)
NORMALIZE_STD = (0.229, 0.224, 0.225)
print(f'Python {sys.version.split()[0]} | torch {torch.__version__} | torchvision {torchvision.__version__}')

## 1. Cấu hình

`data_dir=None` tự tìm dataset trong `/kaggle/input`; nếu có nhiều dataset
FER2013, đặt đường dẫn cụ thể. Mỗi lần chạy tạo một thư mục output riêng.

Chọn `EXPERIMENT` trong cell cấu hình; notebook chỉ train **một** cấu hình
mỗi lần chạy, không tự chạy toàn bộ bảng:

| Preset | Model | MixUp alpha / xác suất | Weight decay |
|---|---|---|---|
| `cbam` (mặc định) | CCT 2 encoder + CBAM | 0.1 / 0.3 | 3e-4 |
| `combined` | CCT 2 encoder, không CBAM | 0.1 / 0.3 | 3e-4 |
| `baseline` | CCT 4 encoder, cấu hình lần trước | Tắt | 1e-4 |
| `mixup_only` | CCT 4 encoder | 0.1 / 0.3 | 1e-4 |
| `two_layers_only` | CCT 2 encoder | Tắt | 1e-4 |
| `weight_decay_only` | CCT 4 encoder | Tắt | 3e-4 |
| `resnet18_baseline` | ResNet18 + Average Pooling + Linear | Tắt | 1e-4 |

`CFG` được tạo từ cấu hình chung rồi cập nhật bởi preset. Nếu muốn tự chỉnh,
đặt `CFG.update(...)` **sau** dòng `CFG.update(EXPERIMENT_PRESETS[EXPERIMENT])`.
`seed=42` điều khiển train; `split_seed=42` giữ nguyên train/validation và mẫu
clean train. Có thể thử seed huấn luyện 42, 43, 44 sau khi chọn cấu hình bằng
validation, giữ `split_seed` cố định để so sánh trên cùng tập ảnh.

Giữ 50 epoch vì lần trước best accuracy xuất hiện muộn ở epoch 38. Chọn best
checkpoint bằng validation accuracy, dùng Macro F1 để phân xử khi accuracy bằng
nhau; chỉ chạy test sau khi chốt cấu hình. Không đổi sang dừng theo val loss
trong lần thử này.

Preset `cbam` chỉ thêm CBAM so với `combined`; giữ nguyên split và cấu hình train.
Checkpoint mới có 2 encoder và CBAM, cần dùng loader theo kiến trúc đã lưu
ở phần cuối notebook. Đổi tên preset không đổi nội dung các tập dữ liệu.

`num_workers=0` đọc dữ liệu trong tiến trình chính để tránh lỗi dọn worker
`AssertionError: can only test a child process` trong notebook Kaggle.
Thiết lập này áp dụng cho train, validation, clean train và test.
Nếu đổi CFG trong một session đã chạy, cần tạo lại các DataLoader vì loader
cũ vẫn giữ thiết lập trước đó. Nếu lỗi dọn worker cũ còn xuất hiện, lưu weights
và khởi động lại session trước lần chạy mới.


**GPU và nạp dữ liệu:** trên Kaggle, `device='cuda'` báo lỗi ngay nếu CUDA
không khả dụng thay vì âm thầm chạy CPU. `cpu_threads=2` giới hạn số luồng
CPU của PyTorch; cần đo trên runtime thực tế để chọn giá trị phù hợp.
`num_workers=0` vẫn cho phép model dùng GPU và tránh lỗi multiprocessing.

`cache_images=True` đọc ảnh RGB gốc vào RAM một lần để giảm mở lại nhiều
file nhỏ ở mỗi epoch. Cache giữ ảnh gốc 48 x 48, không lưu augmentation;
mỗi lần đọc vẫn thực hiện các phép biến đổi ngẫu nhiên. Train và validation
dùng chung cache, test chỉ tạo cache ở bước đánh giá cuối. Nếu pixel cache
đạt `cache_max_mib`, các ảnh còn lại vẫn được đọc từ đĩa như bình thường.
Có thể tắt cache để so sánh tốc độ hoặc giảm RAM.



`cache_workers=4` dùng 4 luồng I/O để mở/decode ảnh song song trong lần
tạo cache. Các luồng này chỉ đọc ảnh gốc, không chạy augmentation và không
tạo tiến trình con. `num_workers=0` của DataLoader được giữ nguyên.
Notebook chỉ gửi một nhóm nhỏ ảnh cho thread pool mỗi lần để giới hạn
số ảnh đã decode đang chờ trong RAM. Đặt `cache_workers=1` để đọc tuần tự.
Tốc độ phụ thuộc nơi lưu dataset; không có mức tăng cố định được bảo đảm.
Khi chạy lại cell tạo dữ liệu hoặc khởi động lại session, cache phải tạo lại.



**Hai GPU T4:** `multi_gpu=True`, `max_gpus=2` bật `nn.DataParallel` khi có
ít nhất hai GPU. `batch_size=64` vẫn là tổng cả hai GPU, thường 32 ảnh/GPU.
Model được nhân bản, mỗi GPU tính một phần batch, sau đó gradient được cộng
để cập nhật cùng bộ trọng số. Nếu chỉ có một GPU hoặc chạy CPU, notebook dùng
một thiết bị. `multi_gpu=False` dùng một GPU để đối chiếu tốc độ.

Lựa chọn DataParallel giữ notebook một tiến trình và không cần torchrun.
PyTorch khuyến nghị DistributedDataParallel cho hiệu năng multi-GPU tốt hơn;
DataParallel phù hợp khi cần tích hợp trực tiếp vào notebook hiện tại.
[Tài liệu PyTorch](https://docs.pytorch.org/docs/stable/generated/torch.nn.DataParallel.html).

Không tăng batch size tự động: giữ nguyên để tránh đồng thời đổi thuật toán
train. BatchNorm của ResNet tính thống kê trên từng phần batch; buffer được
giữ từ GPU chính, nên kết quả có thể khác một GPU dù global batch giữ nguyên.
Không coi đây là chạy đối chứng chỉ đổi tốc độ mà giữ dự đoán giống tuyệt đối.
Hai GPU có thêm chi phí nhân bản/đồng bộ, và CPU vẫn nạp ảnh; tốc độ không
mặc định tăng gấp đôi. Dùng `epoch_seconds` trong history để so thời gian.

Chỉ đổi cấu hình khi bắt đầu run mới. Session đang train không tự chuyển sang
hai GPU; lưu weights hiện có trước khi dừng nếu cần. Checkpoint vẫn không có
tiền tố `module.`, dùng được qua loader trên một GPU hoặc CPU như trước.


In [ ]:
EXPERIMENT = 'cbam'
EXPERIMENT_PRESETS = {
    'cbam': {
        'use_cbam': True, 'model_type': 'cct', 'num_layers': 2, 'mixup_alpha': 0.1,
        'mixup_probability': 0.3, 'weight_decay': 3e-4,
    },
    'baseline': {
        'use_cbam': False, 'model_type': 'cct', 'num_layers': 4, 'mixup_alpha': 0.0,
        'mixup_probability': 0.5, 'weight_decay': 1e-4,
    },
    'mixup_only': {
        'use_cbam': False, 'model_type': 'cct', 'num_layers': 4, 'mixup_alpha': 0.1,
        'mixup_probability': 0.3, 'weight_decay': 1e-4,
    },
    'two_layers_only': {
        'use_cbam': False, 'model_type': 'cct', 'num_layers': 2, 'mixup_alpha': 0.0,
        'mixup_probability': 0.5, 'weight_decay': 1e-4,
    },
    'weight_decay_only': {
        'use_cbam': False, 'model_type': 'cct', 'num_layers': 4, 'mixup_alpha': 0.0,
        'mixup_probability': 0.5, 'weight_decay': 3e-4,
    },
    'resnet18_baseline': {
        'use_cbam': False, 'model_type': 'resnet18', 'num_layers': 4, 'mixup_alpha': 0.0,
        'mixup_probability': 0.5, 'weight_decay': 1e-4,
    },
    'combined': {
        'use_cbam': False, 'model_type': 'cct', 'num_layers': 2, 'mixup_alpha': 0.1,
        'mixup_probability': 0.3, 'weight_decay': 3e-4,
    },
}

CFG = {
    'data_dir': None,
    'output_root': '/kaggle/working' if Path('/kaggle').is_dir() else str(PROJECT_ROOT / 'outputs' / 'kaggle'),
    'run_name': f'fer2013_{EXPERIMENT}',
    'experiment': EXPERIMENT,
    'seed': 42,
    'split_seed': 42,
    'validation_fraction': 0.10,
    'image_size': 224,
    'batch_size': 64,
    'epochs': 50,
    'device': 'cuda' if Path('/kaggle').is_dir() else 'auto',
    'multi_gpu': True,
    'max_gpus': 2,
    'cpu_threads': 2,
    'cache_images': True,
    'cache_workers': 4,
    'cache_max_mib': 256,
    'benchmark_batches': 3,
    'num_workers': 0,  # Đọc ảnh trong tiến trình chính, ổn định trong notebook Kaggle.
    'pretrained': True,
    'backbone_weights_path': None,  # File weights ResNet18 ImageNet khi offline.
    'embed_dim': 512,
    'model_type': 'cct',
    'use_cbam': False,  # Preset cbam bật True; các preset đối chứng tắt.
    'cbam_reduction': 16,
    'cbam_spatial_kernel': 7,
    'num_layers': 2,
    'num_heads': 8,
    'dropout': 0.10,
    'backbone_lr': 3e-5,
    'lr': 2e-4,
    'weight_decay': 3e-4,
    'warmup_epochs': 5,
    'min_lr_ratio': 0.05,
    'loss': 'cross_entropy',
    'label_smoothing': 0.05,
    'focal_gamma': 2.0,
    'erasing_p': 0.10,
    'mixup_alpha': 0.1,
    'mixup_probability': 0.3,
    'gradient_clip': 1.0,
    'use_amp': True,
    'clean_train_every': 5,
    'clean_train_max_samples': 2048,
}


if EXPERIMENT not in EXPERIMENT_PRESETS:
    raise ValueError(f'Preset không hợp lệ: {EXPERIMENT}')
CFG.update(EXPERIMENT_PRESETS[EXPERIMENT])


def validate_config(cfg):
    if cfg['model_type'] not in ('cct', 'resnet18'):
        raise ValueError('model_type phải là cct hoặc resnet18.')
    if not isinstance(cfg['use_cbam'], bool):
        raise ValueError('use_cbam phải là bool.')
    if cfg['use_cbam'] and cfg['model_type'] != 'cct':
        raise ValueError('Preset ResNet18 thuần không dùng CBAM.')
    if not isinstance(cfg['cbam_reduction'], int) or not 1 <= cfg['cbam_reduction'] <= 512:
        raise ValueError('cbam_reduction phải là số nguyên từ 1 đến 512.')
    if cfg['cbam_spatial_kernel'] not in (3, 7):
        raise ValueError('cbam_spatial_kernel phải là 3 hoặc 7.')
    if not 0 <= cfg['seed'] < 2**32 or not 0 <= cfg['split_seed'] < 2**32:
        raise ValueError('seed và split_seed phải nằm trong [0, 2**32).')
    if cfg['embed_dim'] != 512 or cfg['image_size'] != 224:
        raise ValueError('Tokenizer hiện tại yêu cầu embed_dim=512, image_size=224.')
    if cfg['num_layers'] < 1 or cfg['num_heads'] < 1 or 512 % cfg['num_heads']:
        raise ValueError('Số encoder phải dương; 512 phải chia hết cho số heads.')
    if cfg['device'] not in ('auto', 'cpu', 'cuda'):
        raise ValueError('device phải là auto, cpu hoặc cuda.')
    if not isinstance(cfg['multi_gpu'], bool):
        raise ValueError('multi_gpu phải là bool.')
    if isinstance(cfg['max_gpus'], bool) or not isinstance(cfg['max_gpus'], int) or cfg['max_gpus'] < 1:
        raise ValueError('max_gpus phải là số nguyên dương.')
    if cfg['cache_workers'] < 1:
        raise ValueError('cache_workers phải ít nhất bằng 1.')
    if cfg['cpu_threads'] < 1 or cfg['cache_max_mib'] <= 0 or cfg['benchmark_batches'] < 1:
        raise ValueError('cpu_threads, cache_max_mib và benchmark_batches phải dương.')
    if cfg['epochs'] < 1 or cfg['batch_size'] < 2 or cfg['num_workers'] < 0:
        raise ValueError('Cần epochs>=1, batch_size>=2, num_workers>=0.')
    if not 0 <= cfg['warmup_epochs'] < cfg['epochs']:
        raise ValueError('warmup_epochs phải từ 0 đến epochs-1.')
    if not 0 < cfg['validation_fraction'] < 1:
        raise ValueError('validation_fraction phải nằm giữa 0 và 1.')
    if not 0 <= cfg['label_smoothing'] < 1 or cfg['focal_gamma'] < 0:
        raise ValueError('label_smoothing hoặc focal_gamma không hợp lệ.')
    for key in ('erasing_p', 'mixup_probability', 'dropout', 'min_lr_ratio'):
        if not 0 <= cfg[key] <= 1:
            raise ValueError(f'{key} phải nằm trong [0, 1].')
    if cfg['lr'] <= 0 or cfg['backbone_lr'] <= 0 or cfg['weight_decay'] < 0:
        raise ValueError('Learning rate hoặc weight decay không hợp lệ.')
    if cfg['mixup_alpha'] < 0 or cfg['gradient_clip'] <= 0:
        raise ValueError('mixup_alpha hoặc gradient_clip không hợp lệ.')
    if cfg['loss'] not in ('cross_entropy', 'focal'):
        raise ValueError('loss phải là cross_entropy hoặc focal.')
    if cfg['clean_train_every'] < 1 or cfg['clean_train_max_samples'] < 7:
        raise ValueError('Cấu hình đánh giá clean train không hợp lệ.')


def seed_everything(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True


def seed_worker(worker_id):
    worker_seed = torch.initial_seed() % (2 ** 32)
    np.random.seed(worker_seed)
    random.seed(worker_seed)


validate_config(CFG)
def resolve_runtime_device(cfg):
    if cfg['device'] == 'cuda' and not torch.cuda.is_available():
        raise RuntimeError(
            'CUDA chưa khả dụng. Bật GPU trong Kaggle Settings > Accelerator, '
            'khởi động lại session và chạy lại từ đầu. '
            'Đặt device=cpu nếu chủ động muốn dùng CPU.'
        )
    name = cfg['device']
    if name == 'auto':
        name = 'cuda' if torch.cuda.is_available() else 'cpu'
    return torch.device('cuda:0' if name == 'cuda' else name)


torch.set_num_threads(CFG['cpu_threads'])
DEVICE = resolve_runtime_device(CFG)
print('Selected device:', DEVICE, '| CPU threads:', torch.get_num_threads())
if DEVICE.type == 'cuda':
    print('GPU:', torch.cuda.get_device_name(DEVICE), '| Visible GPUs:', torch.cuda.device_count())
    count = min(torch.cuda.device_count(), CFG['max_gpus']) if CFG['multi_gpu'] else 1
    print('Requested execution:', 'DataParallel' if count > 1 else 'single GPU', '| GPU IDs:', list(range(count)))
    print('Batch size tổng:', CFG['batch_size'], '| num_workers chỉ điều khiển CPU đọc ảnh.')
print(json.dumps(CFG, indent=2, ensure_ascii=False))

## 2. Dữ liệu: train / validation / test

Với dataset chỉ có `train/` và `test/`, tách 10% **train** làm validation
theo lớp với seed 42. Train và validation dùng cùng mapping nhãn nhưng
hai bản dataset có transform riêng. Không dùng `test/` trong vòng train.

Nếu có `val/` riêng, notebook dùng trực tiếp tập đó. Các file ảnh không
bị di chuyển; danh sách split được lưu để kiểm tra và tái lập.

In [ ]:
def find_data_root(configured_path=None):
    if configured_path:
        root = Path(configured_path).expanduser().resolve()
        if not (root / 'train').is_dir():
            raise FileNotFoundError(f'Không tìm thấy train/ trong {root}')
        return root
    preferred = [
        Path('/kaggle/input/datasets/msambare/fer2013'),
        Path('/kaggle/input/fer2013'),
        PROJECT_ROOT / 'data' / 'fer2013',
    ]
    for root in preferred:
        if (root / 'train' / 'angry').is_dir():
            return root.resolve()
    input_root = Path('/kaggle/input')
    candidates = set()
    for pattern in ('*/train', '*/*/train', '*/*/*/train'):
        for train_path in input_root.glob(pattern):
            if (train_path / 'angry').is_dir():
                candidates.add(train_path.parent.resolve())
    if len(candidates) == 1:
        return candidates.pop()
    raise FileNotFoundError(
        f'Hãy gắn dataset FER2013 và đặt CFG[data_dir]. Tìm thấy: {sorted(map(str, candidates))}'
    )


def build_transforms(cfg, training=False):
    operations = [
        transforms.Resize((cfg['image_size'], cfg['image_size'])),
        transforms.Grayscale(num_output_channels=3),
    ]
    if training:
        operations += [transforms.RandomHorizontalFlip(), transforms.RandomRotation(10)]
    operations += [
        transforms.ToTensor(),
        transforms.Normalize(NORMALIZE_MEAN, NORMALIZE_STD),
    ]
    if training:
        operations.append(transforms.RandomErasing(
            p=cfg['erasing_p'], scale=(0.02, 0.10), ratio=(0.3, 3.3), value=0,
        ))
    return transforms.Compose(operations)


def check_classes(dataset):
    if dataset.classes != list(CLASS_NAMES):
        raise ValueError(f'Cần các lớp {CLASS_NAMES}, tìm thấy {dataset.classes}')


def iter_decoded_images(paths, workers):
    """Đọc song song theo nhóm nhỏ, giữ nguyên thứ tự và giới hạn ảnh chờ trong RAM."""
    if workers == 1:
        for path in paths:
            yield path, datasets.folder.default_loader(path)
        return
    with ThreadPoolExecutor(max_workers=workers) as pool:
        # Không submit toàn bộ dataset để tránh tích lũy hàng chục nghìn kết quả.
        chunk_size = workers * 2
        for start in range(0, len(paths), chunk_size):
            chunk = paths[start:start + chunk_size]
            for path, image in zip(chunk, pool.map(datasets.folder.default_loader, chunk)):
                yield path, image


class CachedImageLoader:
    """Cache ảnh RGB gốc; dùng luồng I/O, không tạo tiến trình DataLoader con."""

    def __init__(self, paths, max_mib, workers=4):
        self.images = {}
        self.used_bytes = 0
        budget = int(max_mib * 1024 ** 2)
        paths = list(dict.fromkeys(map(str, paths)))
        print(f'Reading {len(paths):,} images with {workers} I/O threads; cache is reused across epochs.')
        with closing(iter_decoded_images(paths, workers)) as decoded, tqdm(
            total=len(paths), desc='Cache decoded images', unit='img',
        ) as progress:
            for path, image in decoded:
                size = image.width * image.height * len(image.getbands())
                if self.used_bytes + size > budget:
                    image.close()
                    break
                self.images[path] = image
                self.used_bytes += size
                progress.update(1)
        print(f'Cached {len(self.images):,} images | RGB pixels {self.used_bytes / 1024**2:.1f} MiB')
        print('Giới hạn trên tính theo pixel RGB; đối tượng PIL/Python dùng thêm một phần RAM.')

    def __call__(self, path):
        path = str(path)
        image = self.images.get(path)
        return image.copy() if image is not None else datasets.folder.default_loader(path)


def attach_image_cache(cfg, *image_datasets):
    if not cfg['cache_images']:
        return None
    paths = (path for dataset in image_datasets for path, _ in dataset.samples)
    cache = CachedImageLoader(paths, cfg['cache_max_mib'], workers=cfg['cache_workers'])
    for dataset in image_datasets:
        dataset.loader = cache
    return cache


def build_data(cfg, root, device):
    root = Path(root).resolve()
    augmented = datasets.ImageFolder(root / 'train', transform=build_transforms(cfg, True))
    clean = datasets.ImageFolder(root / 'train', transform=build_transforms(cfg))
    check_classes(augmented)
    indices = np.arange(len(augmented))
    if (root / 'val').is_dir():
        validation_data = datasets.ImageFolder(root / 'val', transform=build_transforms(cfg))
        check_classes(validation_data)
        train_indices, val_indices = indices, np.arange(len(validation_data))
        validation_source = 'val/'
    else:
        train_indices, val_indices = train_test_split(
            indices, test_size=cfg['validation_fraction'],
            random_state=cfg['split_seed'], stratify=augmented.targets,
        )
        validation_data = clean
        validation_source = 'stratified split from train/'
    attach_image_cache(cfg, augmented, clean, validation_data)
    train_indices = sorted(map(int, train_indices))
    val_indices = sorted(map(int, val_indices))
    train_set = Subset(augmented, train_indices)
    val_set = Subset(validation_data, val_indices)
    train_paths = {augmented.samples[i][0] for i in train_indices}
    val_paths = {validation_data.samples[i][0] for i in val_indices}
    if train_paths & val_paths:
        raise ValueError('Train và validation có đường dẫn trùng nhau.')

    clean_indices = train_indices
    clean_limit = cfg['clean_train_max_samples']
    if clean_limit < len(train_indices):
        _, selected = train_test_split(
            train_indices, test_size=clean_limit, random_state=cfg['split_seed'],
            stratify=np.asarray(clean.targets)[train_indices],
        )
        clean_indices = sorted(map(int, selected))
    clean_set = Subset(clean, clean_indices)
    options = {
        'batch_size': cfg['batch_size'],
        'num_workers': cfg['num_workers'],
        'pin_memory': device.type == 'cuda',
        'worker_init_fn': seed_worker,
    }
    generator = torch.Generator().manual_seed(cfg['seed'])
    loaders = {
        'train': DataLoader(train_set, shuffle=True, generator=generator, **options),
        'val': DataLoader(val_set, shuffle=False, **options),
        'clean_train': DataLoader(clean_set, shuffle=False, **options),
    }
    records = []
    for split, dataset, selected in (
        ('train', augmented, train_indices), ('val', validation_data, val_indices),
    ):
        for index in selected:
            path, label = dataset.samples[index]
            records.append({'split': split, 'path': str(Path(path).relative_to(root)),
                            'label': label, 'class': CLASS_NAMES[label]})
    manifest = pd.DataFrame(records)
    summary = {'train_size': len(train_set), 'val_size': len(val_set),
               'clean_train_size': len(clean_set), 'validation_source': validation_source}
    return loaders, manifest, summary

In [ ]:
seed_everything(CFG['seed'])
DATA_ROOT = find_data_root(CFG['data_dir'])
loaders, split_manifest, data_summary = build_data(CFG, DATA_ROOT, DEVICE)
print('Dataset:', DATA_ROOT)
print(json.dumps(data_summary, indent=2))
print(pd.crosstab(split_manifest['class'], split_manifest['split']).reindex(CLASS_NAMES))
print('test/ được giữ riêng; chỉ đọc ở cell đánh giá cuối.')

## 3. Model: CBAM trước tokenizer

```text
Ảnh B × 3 × 224 × 224
    → ResNet18 feature extractor: B × 512 × 7 × 7
    → Channel Attention: average/max pooling → shared MLP 512 → 32 → 512
    → Spatial Attention: average/max theo kênh → Conv 7 × 7 → sigmoid
    → Flatten + transpose: B × 49 × 512
    → Positional embedding → 2 encoder → SeqPool → Linear 512 → 7
```

Hai attention gate đều nhân trọng số sigmoid vào feature map; CBAM giữ nguyên
kích thước tensor. Phiên bản gọn này dùng Conv không bias và không BatchNorm
trong CBAM, có 32.866 tham số. Cơ chế channel → spatial dựa trên
[CBAM](https://arxiv.org/abs/1807.06521).

CBAM nằm ngoài `tokenizer` trong module để được tối ưu bằng learning rate
head `2e-4`; backbone pretrained vẫn dùng `3e-5`. Khởi tạo encoder độc lập,
không reset backbone pretrained. Các preset không CBAM dùng `Identity` tại
vị trí này; lớp model mặc định `use_cbam=False` để nạp checkpoint cũ.

Nếu offline, gắn weights ResNet18 và đặt `backbone_weights_path`. Notebook
không tự chuyển sang random weights khi tải lỗi. Train CBAM từ đầu trong run
mới; không nạp trực tiếp raw weights không CBAM bằng `strict=False`.

In [ ]:
class ChannelAttention(nn.Module):
    def __init__(self, channels=512, reduction=16):
        super().__init__()
        if channels < 1 or not isinstance(reduction, int) or not 1 <= reduction <= channels:
            raise ValueError('Cần channels >= 1 và reduction nguyên trong [1, channels].')
        hidden = max(1, channels // reduction)
        self.shared_mlp = nn.Sequential(
            nn.Conv2d(channels, hidden, kernel_size=1, bias=False),
            nn.ReLU(),
            nn.Conv2d(hidden, channels, kernel_size=1, bias=False),
        )

    def forward(self, x):
        average = F.adaptive_avg_pool2d(x, 1)
        maximum = F.adaptive_max_pool2d(x, 1)
        weights = torch.sigmoid(self.shared_mlp(average) + self.shared_mlp(maximum))
        return x * weights


class SpatialAttention(nn.Module):
    def __init__(self, kernel_size=7):
        super().__init__()
        if kernel_size not in (3, 7):
            raise ValueError('Spatial kernel phải là 3 hoặc 7.')
        self.conv = nn.Conv2d(2, 1, kernel_size=kernel_size,
                              padding=kernel_size // 2, bias=False)

    def forward(self, x):
        average = x.mean(dim=1, keepdim=True)
        maximum = x.amax(dim=1, keepdim=True)
        weights = torch.sigmoid(self.conv(torch.cat((average, maximum), dim=1)))
        return x * weights


class CBAM(nn.Module):
    def __init__(self, channels=512, reduction=16, spatial_kernel=7):
        super().__init__()
        self.channel_attention = ChannelAttention(channels, reduction)
        self.spatial_attention = SpatialAttention(spatial_kernel)

    def forward(self, x):
        return self.spatial_attention(self.channel_attention(x))


class ResNetTokenizer(nn.Module):
    def __init__(self, pretrained=True, backbone_weights_path=None):
        super().__init__()
        weights = models.ResNet18_Weights.IMAGENET1K_V1 if pretrained and not backbone_weights_path else None
        try:
            backbone = models.resnet18(weights=weights)
        except Exception as exc:
            raise RuntimeError(
                'Không tải được weights ResNet18. Bật Internet hoặc gắn file weights '
                'và đặt backbone_weights_path. Chọn pretrained=False nếu muốn train từ đầu.'
            ) from exc
        if backbone_weights_path:
            state = torch.load(backbone_weights_path, map_location='cpu', weights_only=True)
            backbone.load_state_dict(state, strict=True)
        self.feature_extractor = nn.Sequential(*list(backbone.children())[:-2])

    def forward_features(self, x):
        return self.feature_extractor(x)

    def forward(self, x):
        return self.forward_features(x).flatten(2).transpose(1, 2)


class SeqPool(nn.Module):
    def __init__(self, embedding_dim):
        super().__init__()
        self.attention_pool = nn.Linear(embedding_dim, 1)

    def forward(self, x):
        weights = self.attention_pool(x).softmax(dim=1)
        return torch.matmul(weights.transpose(1, 2), x).squeeze(1)


class CCT_ResNet(nn.Module):
    def __init__(self, num_classes=7, embed_dim=512, num_layers=2, num_heads=8,
                 pretrained=True, dropout=0.1, backbone_weights_path=None,
                 use_cbam=False, cbam_reduction=16, cbam_spatial_kernel=7):
        super().__init__()
        if embed_dim != 512:
            raise ValueError('ResNet18 trả về 512 kênh; cần projection để đổi embed_dim.')
        self.tokenizer = ResNetTokenizer(pretrained, backbone_weights_path)
        # CBAM là lớp mới: đặt ngoài tokenizer để dùng learning rate của head.
        self.cbam = CBAM(embed_dim, cbam_reduction, cbam_spatial_kernel) if use_cbam else nn.Identity()
        self.positional_emb = nn.Parameter(torch.empty(1, 49, embed_dim))
        encoder = nn.TransformerEncoderLayer(
            d_model=embed_dim, nhead=num_heads, dim_feedforward=embed_dim * 4,
            dropout=dropout, activation='gelu', batch_first=True,
        )
        self.transformer = nn.TransformerEncoder(encoder, num_layers=num_layers)
        self.seq_pool = SeqPool(embed_dim)
        self.classifier = nn.Linear(embed_dim, num_classes)
        self.initialize_new_layers()

    def initialize_new_layers(self):
        nn.init.normal_(self.positional_emb, std=0.02)
        for encoder in self.transformer.layers:
            nn.init.xavier_uniform_(encoder.self_attn.in_proj_weight)
            nn.init.zeros_(encoder.self_attn.in_proj_bias)
            for module in encoder.modules():
                if isinstance(module, nn.Linear):
                    nn.init.xavier_uniform_(module.weight)
                    if module.bias is not None:
                        nn.init.zeros_(module.bias)
                elif isinstance(module, nn.LayerNorm):
                    nn.init.ones_(module.weight)
                    nn.init.zeros_(module.bias)
        for module in (self.seq_pool.attention_pool, self.classifier):
            nn.init.xavier_uniform_(module.weight)
            nn.init.zeros_(module.bias)

    def forward(self, x):
        features = self.tokenizer.forward_features(x)
        features = self.cbam(features)
        tokens = features.flatten(2).transpose(1, 2)
        if tokens.shape[1:] != self.positional_emb.shape[1:]:
            raise ValueError('Đầu vào cần tạo 49 token x 512 chiều; dùng ảnh 224 x 224.')
        tokens = self.transformer(tokens + self.positional_emb)
        return self.classifier(self.seq_pool(tokens))


class ResNet18Baseline(nn.Module):
    def __init__(self, num_classes=7, pretrained=True, backbone_weights_path=None):
        super().__init__()
        self.tokenizer = ResNetTokenizer(pretrained, backbone_weights_path)
        self.classifier = nn.Linear(512, num_classes)
        nn.init.xavier_uniform_(self.classifier.weight)
        nn.init.zeros_(self.classifier.bias)

    def forward(self, x):
        return self.classifier(self.tokenizer(x).mean(dim=1))


def model_config_from_cfg(cfg):
    spec = {'model_type': cfg['model_type'], 'num_classes': len(CLASS_NAMES)}
    if cfg['model_type'] == 'cct':
        spec.update(embed_dim=cfg['embed_dim'], num_layers=cfg['num_layers'],
                    num_heads=cfg['num_heads'], dropout=cfg['dropout'],
                    use_cbam=cfg['use_cbam'], cbam_reduction=cfg['cbam_reduction'],
                    cbam_spatial_kernel=cfg['cbam_spatial_kernel'])
    return spec


def build_model_from_spec(spec, pretrained=False, backbone_weights_path=None):
    options = {key: value for key, value in spec.items() if key != 'model_type'}
    if spec['model_type'] == 'cct':
        return CCT_ResNet(**options, pretrained=pretrained,
                          backbone_weights_path=backbone_weights_path)
    if spec['model_type'] == 'resnet18':
        return ResNet18Baseline(**options, pretrained=pretrained,
                                backbone_weights_path=backbone_weights_path)
    raise ValueError(f'Kiến trúc không hỗ trợ: {spec["model_type"]}')


def build_model(cfg, pretrained=None):
    use_pretrained = cfg['pretrained'] if pretrained is None else pretrained
    return build_model_from_spec(
        model_config_from_cfg(cfg), pretrained=use_pretrained,
        backbone_weights_path=cfg['backbone_weights_path'] if use_pretrained else None,
    )


def unwrap_model(model):
    return model.module if isinstance(model, nn.DataParallel) else model


def execution_info(model, device):
    if isinstance(model, nn.DataParallel) and model.device_ids:
        return {'strategy': 'DataParallel', 'device_ids': list(model.device_ids)}
    ids = [0 if device.index is None else device.index] if device.type == 'cuda' else []
    return {'strategy': 'single', 'device_ids': ids}


def prepare_parallel_model(model, cfg, device):
    model = unwrap_model(model).to(device)
    if device.type != 'cuda':
        return model
    count = min(torch.cuda.device_count(), cfg.get('max_gpus', 2)) if cfg.get('multi_gpu', False) else 1
    if count < 2:
        if cfg.get('multi_gpu', False):
            print('Chỉ có một GPU được dùng; chạy tiếp bằng single GPU.')
        return model
    if device.index not in (None, 0):
        raise ValueError('DataParallel trong notebook này cần thiết bị chính cuda:0.')
    if cfg['batch_size'] < count:
        raise ValueError('batch_size tổng phải ít nhất bằng số GPU.')
    ids = list(range(count))
    parallel = nn.DataParallel(model, device_ids=ids, output_device=ids[0])
    print(f'DataParallel enabled: GPU IDs {ids} | global batch {cfg["batch_size"]}')
    for index in ids:
        print(f'  cuda:{index}: {torch.cuda.get_device_name(index)}')
    return parallel


def synchronize_model_devices(model, device):
    if device.type == 'cuda':
        for index in execution_info(model, device)['device_ids']:
            torch.cuda.synchronize(index)


def checkpoint_paths(cfg, run_dir):
    prefix = 'cct_resnet' if cfg['model_type'] == 'cct' else 'resnet18'
    if cfg['model_type'] == 'cct' and cfg['use_cbam']:
        prefix += '_cbam'
    stem = f'{prefix}_fer2013_best'
    return Path(run_dir) / f'{stem}.pth', Path(run_dir) / f'{stem}_checkpoint.pth'


def checkpoint_payload(model, cfg, epoch, metrics):
    return {
        'format_version': 1,
        'state_dict': unwrap_model(model).state_dict(),
        'model_config': model_config_from_cfg(cfg),
        'classes': list(CLASS_NAMES),
        'preprocess': {'image_size': cfg['image_size'],
                       'mean': list(NORMALIZE_MEAN), 'std': list(NORMALIZE_STD)},
        'epoch': epoch,
        'validation': {key: metrics[key] for key in ('loss', 'accuracy', 'macro_f1')},
    }


def load_trained_model(checkpoint, device):
    payload = torch.load(checkpoint, map_location='cpu', weights_only=True)
    if not isinstance(payload, dict) or payload.get('format_version') != 1:
        raise ValueError('Cần file *_checkpoint.pth có kèm model_config, không phải raw weights.')
    if payload['classes'] != list(CLASS_NAMES):
        raise ValueError('Thứ tự nhãn trong checkpoint không khớp notebook.')
    expected = {'image_size': 224, 'mean': list(NORMALIZE_MEAN), 'std': list(NORMALIZE_STD)}
    if payload['preprocess'] != expected or payload['model_config']['num_classes'] != len(CLASS_NAMES):
        raise ValueError('Tiền xử lý hoặc số lớp trong checkpoint không khớp notebook.')
    restored = build_model_from_spec(payload['model_config'], pretrained=False)
    restored.load_state_dict(payload['state_dict'], strict=True)
    return restored.to(device).eval()

## 4. Loss, optimizer và warmup

Cross-Entropy là mặc định để kiểm tra một cấu hình đơn giản hơn Focal Loss.
Learning rate tăng tuyến tính trong warmup rồi giảm cosine xuống 5% mức gốc.
Xác suất Random Erasing giữ cố định; không phụ thuộc kết quả validation.

In [ ]:
class FocalLoss(nn.Module):
    def __init__(self, gamma=2.0, label_smoothing=0.05):
        super().__init__()
        self.gamma = gamma
        self.label_smoothing = label_smoothing

    def forward(self, logits, targets):
        ce = F.cross_entropy(logits, targets, reduction='none')
        smooth_ce = F.cross_entropy(
            logits, targets, reduction='none', label_smoothing=self.label_smoothing,
        )
        return ((1 - torch.exp(-ce)).pow(self.gamma) * smooth_ce).mean()


def build_criterion(cfg):
    if cfg['loss'] == 'focal':
        return FocalLoss(cfg['focal_gamma'], cfg['label_smoothing'])
    return nn.CrossEntropyLoss(label_smoothing=cfg['label_smoothing'])


def lr_multiplier(epoch_index, cfg):
    warmup = cfg['warmup_epochs']
    if epoch_index < warmup:
        return (epoch_index + 1) / warmup
    span = max(1, cfg['epochs'] - warmup - 1)
    progress = min(1.0, max(0.0, (epoch_index - warmup) / span))
    return cfg['min_lr_ratio'] + (1 - cfg['min_lr_ratio']) * 0.5 * (1 + math.cos(math.pi * progress))


def build_optimizer(model, cfg):
    model = unwrap_model(model)
    groups = [
        {'params': model.tokenizer.parameters(), 'lr': cfg['backbone_lr'], 'name': 'backbone'},
        {'params': [p for name, p in model.named_parameters() if not name.startswith('tokenizer.')],
         'lr': cfg['lr'], 'name': 'head'},
    ]
    optimizer = torch.optim.AdamW(groups, weight_decay=cfg['weight_decay'])
    scheduler = torch.optim.lr_scheduler.LambdaLR(
        optimizer, lr_lambda=lambda epoch_index: lr_multiplier(epoch_index, cfg),
    )
    return optimizer, scheduler


def amp_context(device, enabled):
    return torch.autocast(device_type='cuda', dtype=torch.float16) if enabled and device.type == 'cuda' else nullcontext()

## 5. Train và đánh giá

`train_accuracy` được tính trên ảnh augmentation; nếu bật MixUp thì tính theo
trọng số hai nhãn. `clean_train_accuracy` đo trên một mẫu train cố định,
không augmentation/MixUp, tối đa 2048 ảnh, mỗi 5 epoch và ở epoch cuối.
Nó giúp chẩn đoán khả năng học mà không nhầm với accuracy trên ảnh trộn.

Tất cả accuracy/F1 trong CSV và JSON có thang **0–1**; khi in ra màn hình
được đổi thành phần trăm. Loss mặc định mới là Cross-Entropy, nên không
so trực tiếp giá trị loss với log Focal Loss trước đây.

In [ ]:
def train_one_epoch(model, loader, optimizer, criterion, scaler, device, cfg, epoch):
    model.train()
    total = correct = loss_total = 0.0
    use_amp = cfg['use_amp'] and device.type == 'cuda'
    progress = tqdm(loader, desc=f'Epoch {epoch}/{cfg["epochs"]} [Train]')
    for inputs, targets in progress:
        inputs = inputs.to(device, non_blocking=True)
        targets = targets.to(device, non_blocking=True)
        lam, paired_targets = 1.0, targets
        if cfg['mixup_alpha'] > 0 and np.random.random() < cfg['mixup_probability']:
            lam = float(np.random.beta(cfg['mixup_alpha'], cfg['mixup_alpha']))
            index = torch.randperm(inputs.size(0), device=device)
            inputs = lam * inputs + (1 - lam) * inputs[index]
            paired_targets = targets[index]
        optimizer.zero_grad(set_to_none=True)
        with amp_context(device, use_amp):
            logits = model(inputs)
            loss = lam * criterion(logits, targets) + (1 - lam) * criterion(logits, paired_targets)
        if not torch.isfinite(loss).item():
            raise FloatingPointError(f'Loss không hữu hạn ở epoch {epoch}.')
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        nn.utils.clip_grad_norm_(model.parameters(), cfg['gradient_clip'])
        scaler.step(optimizer)
        scaler.update()
        predicted = logits.detach().argmax(dim=1)
        correct += lam * predicted.eq(targets).sum().item()
        correct += (1 - lam) * predicted.eq(paired_targets).sum().item()
        total += targets.size(0)
        loss_total += loss.item() * targets.size(0)
        progress.set_postfix(loss=f'{loss.item():.4f}')
    return {'loss': loss_total / total, 'accuracy': correct / total}


@torch.inference_mode()
def evaluate(model, loader, device, criterion=None, use_amp=False):
    model.eval()
    targets_all, predictions_all = [], []
    loss_total = 0.0
    for inputs, targets in tqdm(loader, desc='Evaluate', leave=False):
        inputs = inputs.to(device, non_blocking=True)
        targets = targets.to(device, non_blocking=True)
        with amp_context(device, use_amp):
            logits = model(inputs)
            loss = criterion(logits, targets) if criterion is not None else None
        if not torch.isfinite(logits).all().item():
            raise FloatingPointError('Logits không hữu hạn trong evaluate.')
        if loss is not None:
            loss_total += loss.item() * targets.size(0)
        targets_all.extend(targets.cpu().tolist())
        predictions_all.extend(logits.argmax(dim=1).cpu().tolist())
    if not targets_all:
        raise ValueError('Tập đánh giá rỗng.')
    return {
        'loss': loss_total / len(targets_all) if criterion is not None else None,
        'accuracy': float(np.mean(np.asarray(targets_all) == predictions_all)),
        'macro_f1': float(f1_score(targets_all, predictions_all,
                                 labels=list(range(len(CLASS_NAMES))), average='macro', zero_division=0)),
        'targets': targets_all, 'predictions': predictions_all,
    }


def scalar_metrics(metrics):
    return {key: value for key, value in metrics.items() if key not in ('targets', 'predictions')}


def run_training(cfg, loaders, manifest, summary, root, device):
    validate_config(cfg)
    torch.set_num_threads(cfg['cpu_threads'])
    expected_device = resolve_runtime_device(cfg)
    if expected_device.type != device.type:
        raise ValueError('Thiết bị đã đổi; chạy lại cell cấu hình và cell tạo DataLoader.')
    seed_everything(cfg['seed'])
    if loaders['train'].generator is not None:
        loaders['train'].generator.manual_seed(cfg['seed'])
    stamp = datetime.now(timezone.utc).strftime('%Y%m%d_%H%M%S_%f')
    run_dir = Path(cfg['output_root']) / f'{cfg["run_name"]}_{stamp}'
    run_dir.mkdir(parents=True, exist_ok=False)
    manifest.to_csv(run_dir / 'split_manifest.csv', index=False)
    model = prepare_parallel_model(build_model(cfg), cfg, device)
    print('Model parameters device:', next(model.parameters()).device)
    print('Execution:', execution_info(model, device))
    print('AMP:', cfg['use_amp'] and device.type == 'cuda', '| DataLoader workers:', loaders['train'].num_workers)
    criterion = build_criterion(cfg)
    optimizer, scheduler = build_optimizer(model, cfg)
    use_amp = cfg['use_amp'] and device.type == 'cuda'
    scaler = torch.amp.GradScaler('cuda', enabled=use_amp)
    checkpoint_path, bundle_path = checkpoint_paths(cfg, run_dir)
    best_accuracy, best_macro_f1, best_epoch = -1.0, -1.0, None
    history = []
    print('Output:', run_dir)
    print('Parameters:', f'{sum(p.numel() for p in model.parameters()):,}')
    metadata = {
        'config': cfg, 'data_root': str(root), 'classes': list(CLASS_NAMES),
        'data_summary': summary, 'torch_version': torch.__version__,
        'torchvision_version': torchvision.__version__,
        'device': str(device), 'metric_scale': '0 to 1',
        'parameter_count': sum(p.numel() for p in model.parameters()),
        'execution': execution_info(model, device),
        'global_batch_size': cfg['batch_size'],
        'model_config': model_config_from_cfg(cfg),
        'weights_file': checkpoint_path.name, 'checkpoint_file': bundle_path.name,
    }
    (run_dir / 'training.json').write_text(json.dumps(metadata, indent=2), encoding='utf-8')
    (run_dir / 'model_config.json').write_text(json.dumps(model_config_from_cfg(cfg), indent=2), encoding='utf-8')
    print('Model config:', model_config_from_cfg(cfg))
    print('MixUp alpha/probability:', cfg['mixup_alpha'], cfg['mixup_probability'], '| Weight decay:', cfg['weight_decay'])
    for epoch in range(1, cfg['epochs'] + 1):
        epoch_started = perf_counter()
        rates = {group['name']: group['lr'] for group in optimizer.param_groups}
        training = train_one_epoch(model, loaders['train'], optimizer, criterion, scaler, device, cfg, epoch)
        validation = evaluate(model, loaders['val'], device, criterion, use_amp)
        clean = None
        if epoch % cfg['clean_train_every'] == 0 or epoch == cfg['epochs']:
            clean = evaluate(model, loaders['clean_train'], device, use_amp=use_amp)
        row = {
            'epoch': epoch, 'train_loss': training['loss'], 'train_accuracy': training['accuracy'],
            'val_loss': validation['loss'], 'val_accuracy': validation['accuracy'],
            'val_macro_f1': validation['macro_f1'],
            'clean_train_accuracy': clean['accuracy'] if clean else None,
            'lr_backbone': rates['backbone'], 'lr_head': rates['head'],
            'epoch_seconds': perf_counter() - epoch_started,
        }
        history.append(row)
        pd.DataFrame(history).to_csv(run_dir / 'history.csv', index=False)
        improved = validation['accuracy'] > best_accuracy or (
            validation['accuracy'] == best_accuracy and validation['macro_f1'] > best_macro_f1
        )
        if improved:
            best_accuracy, best_macro_f1, best_epoch = validation['accuracy'], validation['macro_f1'], epoch
            torch.save(unwrap_model(model).state_dict(), checkpoint_path)
            torch.save(checkpoint_payload(model, cfg, epoch, validation), bundle_path)
            best = {'epoch': best_epoch, **scalar_metrics(validation)}
            (run_dir / 'best_validation.json').write_text(json.dumps(best, indent=2), encoding='utf-8')
        print(
            f'Epoch {epoch:02d} | Train loss {training["loss"]:.4f} | Val loss {validation["loss"]:.4f} | '
            f'Train acc {training["accuracy"]:.2%} | Val acc {validation["accuracy"]:.2%} | '
            f'Val macro F1 {validation["macro_f1"]:.2%} | LR {rates["backbone"]:.2e}/{rates["head"]:.2e} | '
            f'Time {row["epoch_seconds"]:.1f}s'
        )
        if clean:
            print(f'Clean train accuracy ({len(loaders["clean_train"].dataset)} ảnh): {clean["accuracy"]:.2%}')
        if improved:
            print(f'Saved best checkpoint: epoch {best_epoch}, Val acc {best_accuracy:.2%}')
        scheduler.step()
    state = torch.load(checkpoint_path, map_location='cpu', weights_only=True)
    model = unwrap_model(model)
    model.load_state_dict(state, strict=True)
    model.eval()
    print(f'Reloaded best checkpoint: epoch {best_epoch}, Val acc {best_accuracy:.2%}')
    return model, pd.DataFrame(history), run_dir

### Kiểm tra GPU và tốc độ trước khi train

Cell sau đo thời gian nạp batch và thời gian một bước train bằng **model thử**.
Model thử có cùng kiến trúc, khởi tạo ngẫu nhiên để không tải weights lần nữa,
rồi được bỏ đi. Model dùng để train thật được tạo mới ở cell tiếp theo.

Nếu `Model/Batch device` là `cuda:0`, phép tính đang dùng GPU dù biểu đồ
có lúc hiện 0%. Nạp/resize/augmentation ảnh vẫn dùng CPU. Nếu load chậm hơn
train step, CPU vẫn có thể là điểm nghẽn ngay cả khi cả hai GPU tham gia.
Kiểm tra `Benchmark execution` để xem GPU IDs thực tế; DataParallel sẽ in `[0, 1]`.

Việc đo có đồng bộ tất cả GPU tham gia để tránh thời gian tính toán bị ghi nhận quá thấp.
Run thật sẽ reset seed và generator, nên benchmark không tiêu thụ thứ tự train.

In [ ]:
def benchmark_runtime(cfg, loader, device):
    """Đo nạp dữ liệu thực và một bước train, không thay đổi model đang huấn luyện."""
    count = min(cfg['benchmark_batches'], len(loader))
    if count < 1:
        raise ValueError('DataLoader rỗng.')
    probe_cfg = {**cfg, 'pretrained': False, 'backbone_weights_path': None}
    probe = prepare_parallel_model(build_model(probe_cfg), cfg, device)
    optimizer, _ = build_optimizer(probe, probe_cfg)
    use_amp = cfg['use_amp'] and device.type == 'cuda'
    scaler = torch.amp.GradScaler('cuda', enabled=use_amp)
    criterion = build_criterion(cfg)
    print('Benchmark model device:', next(probe.parameters()).device)
    print('Benchmark execution:', execution_info(probe, device))
    print('CUDA available:', torch.cuda.is_available(), '| Workers:', loader.num_workers)
    if device.type == 'cuda':
        print('GPU:', torch.cuda.get_device_name(device))

    def train_step(inputs, targets):
        optimizer.zero_grad(set_to_none=True)
        with amp_context(device, use_amp):
            logits = probe(inputs)
            loss = criterion(logits, targets)
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        nn.utils.clip_grad_norm_(probe.parameters(), cfg['gradient_clip'])
        scaler.step(optimizer)
        scaler.update()

    iterator = iter(loader)
    samples = []
    try:
        # Đo CPU loading riêng; không giữ nhiều batch float trong RAM/GPU.
        for index in range(count):
            started = perf_counter()
            inputs, targets = next(iterator)
            load_seconds = perf_counter() - started
            inputs = inputs.to(device, non_blocking=True)
            targets = targets.to(device, non_blocking=True)
            if index == 0:
                print('Batch device:', inputs.device)
                train_step(inputs, targets)  # Warmup, bỏ thời gian khởi tạo CUDA/optimizer.
            synchronize_model_devices(probe, device)
            started = perf_counter()
            train_step(inputs, targets)
            synchronize_model_devices(probe, device)
            step_seconds = perf_counter() - started
            samples.append({'batch': index + 1, 'load_seconds': load_seconds,
                            'train_step_seconds': step_seconds})
        report = pd.DataFrame(samples)
        print(report.to_string(index=False))
        mean_load = report['load_seconds'].mean()
        mean_step = report['train_step_seconds'].mean()
        print(f'Mean load: {mean_load:.3f}s | Mean train step: {mean_step:.3f}s')
        if mean_load > mean_step:
            print('Nạp/biến đổi ảnh đang mất nhiều thời gian hơn bước tính toán model.')
        else:
            print('Bước tính toán model đang mất nhiều thời gian hơn nạp ảnh.')
        return report
    finally:
        del iterator, optimizer, probe
        if device.type == 'cuda':
            torch.cuda.empty_cache()


# Chạy cell này trước train; có thể bỏ qua ở lần chạy sau nếu không cần đo lại.
runtime_report = benchmark_runtime(CFG, loaders['train'], DEVICE)

In [ ]:
model, history, RUN_DIR = run_training(
    CFG, loaders, split_manifest, data_summary, DATA_ROOT, DEVICE,
)

## 6. Lịch sử huấn luyện

Clean train accuracy dùng ảnh không augmentation và model ở chế độ eval.
Đây là accuracy trên một mẫu train cố định, không phải kết quả test.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 4))
history.plot(x='epoch', y=['train_loss', 'val_loss'], ax=axes[0], title='Loss')
history.plot(x='epoch', y=['train_accuracy', 'val_accuracy', 'val_macro_f1'],
             ax=axes[1], title='Accuracy / Macro F1 (0-1)')
clean_points = history.dropna(subset=['clean_train_accuracy'])
axes[1].plot(clean_points['epoch'], clean_points['clean_train_accuracy'],
             'o--', label='clean_train_accuracy')
axes[1].legend()
history.plot(x='epoch', y=['lr_backbone', 'lr_head'], ax=axes[2], title='Learning rate')
fig.tight_layout()
fig.savefig(RUN_DIR / 'training_history.png', dpi=150)
plt.show()

## 7. Đánh giá cuối trên test

Chỉ chạy bước này sau khi đã chốt cấu hình bằng validation.
Cell nạp lại best checkpoint, rồi mới mở `test/`; kết quả test không được dùng
để chọn epoch hoặc điều chỉnh augmentation trong vòng train.

Số lượng ảnh test được in và lưu. Một số bản FER2013 dạng thư mục có thể gộp
PublicTest và PrivateTest; cần xác nhận nguồn split trước khi so với các paper
sử dụng riêng PrivateTest 3589 ảnh. Chỉ tên thư mục `test` chưa xác nhận điều đó.

Nếu tập test này đã được dùng để chọn model ở lần chạy cũ, kết quả mới vẫn
hữu ích để tham khảo, nhưng không xóa được việc đã tham khảo test trước đây.

In [ ]:
def evaluate_final_test(model, cfg, root, run_dir, device):
    metadata = json.loads((Path(run_dir) / 'training.json').read_text(encoding='utf-8'))
    cfg = metadata['config']
    model = load_trained_model(Path(run_dir) / metadata['checkpoint_file'], device)
    model = prepare_parallel_model(model, cfg, device)
    print('Test execution:', execution_info(model, device))
    test_dataset = datasets.ImageFolder(Path(root) / 'test', transform=build_transforms(cfg))
    check_classes(test_dataset)
    attach_image_cache(cfg, test_dataset)
    test_loader = DataLoader(
        test_dataset, batch_size=cfg['batch_size'], shuffle=False,
        num_workers=cfg['num_workers'], pin_memory=device.type == 'cuda',
        worker_init_fn=seed_worker,
    )
    result = evaluate(model, test_loader, device, build_criterion(cfg),
                      cfg['use_amp'] and device.type == 'cuda')
    metrics = {**scalar_metrics(result), 'test_size': len(test_dataset)}
    (Path(run_dir) / 'test_metrics.json').write_text(json.dumps(metrics, indent=2), encoding='utf-8')
    print('Test images:', len(test_dataset))
    print(f'Test accuracy: {result["accuracy"]:.2%} | Test macro F1: {result["macro_f1"]:.2%}')
    report = classification_report(
        result['targets'], result['predictions'], labels=list(range(len(CLASS_NAMES))),
        target_names=list(CLASS_NAMES), digits=4, zero_division=0,
    )
    print(report)
    (Path(run_dir) / 'classification_report.txt').write_text(report, encoding='utf-8')
    pd.DataFrame({
        'path': [str(Path(path).relative_to(root)) for path, _ in test_dataset.samples],
        'target': result['targets'], 'prediction': result['predictions'],
    }).to_csv(Path(run_dir) / 'test_predictions.csv', index=False)
    matrix = confusion_matrix(result['targets'], result['predictions'], labels=list(range(len(CLASS_NAMES))))
    pd.DataFrame(matrix, index=CLASS_NAMES, columns=CLASS_NAMES).to_csv(Path(run_dir) / 'confusion_matrix.csv')
    return result, matrix


test_result, matrix = evaluate_final_test(model, CFG, DATA_ROOT, RUN_DIR, DEVICE)
fig, ax = plt.subplots(figsize=(9, 7))
sns.heatmap(matrix, annot=True, fmt='d', cmap='Blues', ax=ax,
            xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES)
ax.set(title='FER2013 test — best validation checkpoint', xlabel='Predicted', ylabel='True')
fig.tight_layout()
fig.savefig(RUN_DIR / 'test_confusion_matrix.png', dpi=150)
plt.show()

## 8. Tải kết quả và nạp lại model

Run mặc định tạo thư mục `fer2013_cbam_<thời gian>` trong `/kaggle/working`.
Đối chiếu đường dẫn `Output:` in đầu cell train để lấy đúng lần chạy.

- `cct_resnet_cbam_fer2013_best.pth`: raw weights tốt nhất.
- `cct_resnet_cbam_fer2013_best_checkpoint.pth`: weights kèm kiến trúc,
  thông số CBAM, nhãn, tiền xử lý, best epoch và validation metrics.
- `model_config.json`, `training.json`, `best_validation.json`, `history.csv`.
- `split_manifest.csv`: danh sách ảnh train/validation để tái lập split.
- Các báo cáo test, dự đoán và confusion matrix sau bước đánh giá cuối.

Preset `combined` và các CCT không CBAM giữ tên `cct_resnet_fer2013_best...`;
ResNet18 thuần dùng tiền tố `resnet18`. Cell test đọc tên file từ metadata,
không suy đoán theo thư mục hay preset hiện đang chọn.

Sau các cell định nghĩa model, nạp lại mà không tải weights ImageNet:

```python
restored_model = load_trained_model(
    RUN_DIR / 'cct_resnet_cbam_fer2013_best_checkpoint.pth', DEVICE,
)
```

Hàm nạp vẫn đọc được các file `*_checkpoint.pth` có metadata của run combined
trước khi thêm CBAM. Raw state dict cũ cần tạo đúng kiến trúc trước khi nạp.
Ứng dụng hiện tại trong `src/fer` chưa có CBAM nên chưa dùng trực tiếp
`--weights` với model CBAM; dùng các định nghĩa và loader trong notebook này.

Checkpoint chỉ lưu trọng số tốt nhất khi validation cải thiện, không lưu model
của mỗi batch. File không chứa optimizer/scheduler để resume; chạy lại cell
train tạo một run mới. So `cbam` với `combined` trên cùng split seed và train
seed, chọn cấu hình bằng validation rồi đánh giá test sau khi chốt.


Khi train DataParallel, cả raw weights và checkpoint có metadata đều lưu
`model.module.state_dict()` để giữ tên tensor như khi train một GPU.
`training.json` ghi `execution.strategy`, GPU IDs và global batch size của run;
số GPU không phải thuộc tính kiến trúc và không bắt buộc phải khớp khi nạp.
